# Fase 4 + 5 (Gabungan) — Training & Evaluasi IndoBERT (Sentara)

Notebook **Run-All** satu sesi untuk Google Colab (GPU). Menggabungkan:
- **Fase 4**: full fine-tuning IndoBERT dengan konfigurasi final (LR 3e-5, batch 8, **2 epoch** — anti-overfitting per learning curve) + ekspor loss per epoch.
- **Fase 5**: evaluasi test set, cek overfitting, confusion matrix, learning curve, gate.

> **Sengaja DILEWATI untuk hemat GPU** (sudah selesai & ter-commit di run Fase 4 sebelumnya): focused random search (FR-4.6) dan 5-fold cross-validation (FR-4.7). Ringkasan CV tetap dipakai di laporan Fase 5 via `cross_validation_report.json`.

**Cara pakai:** set Runtime ▸ T4 GPU, lalu **Runtime ▸ Run all**. Estimasi ~25–45 menit (mayoritas di training).

Metrik utama: **macro F1** (target ≥ 0.85). Class weight wajib (imbalance kelas > 15%).

## 1. Setup environment
Clone repo + install dependensi (pinning `transformers<5` agar API Trainer konsisten).

In [ ]:
# Clone repo proyek
!git clone https://github.com/rahmatullahaditya780/sentiment-analysis-indobert.git
%cd sentiment-analysis-indobert

!pip -q install "transformers>=4.40,<5" "torch>=2.2.0" "datasets>=2.19.0" "accelerate>=0.30.0" "scikit-learn>=1.4.0" "pandas>=2.2.0" "matplotlib>=3.9.0"

# Alternatif (mount Drive, bila tak via GitHub):
# from google.colab import drive; drive.mount("/content/drive")
# %cd /content/drive/MyDrive/SKRIPSI/sentiment-analysis-indobert

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('CUDA tersedia:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Load data bersih (output Fase 3)
`clean_train/validation/test.csv` (kolom: text, label, source).

In [ ]:
from src.modeling.data import load_clean_split, label_distribution, compute_class_weights
from src.modeling.config import LABEL2ID

train_df = load_clean_split('train')
val_df = load_clean_split('validation')
test_df = load_clean_split('test')

print('train/val/test:', len(train_df), len(val_df), len(test_df))
print('LABEL2ID:', LABEL2ID)
print('Distribusi train:', label_distribution(train_df))
print('Class weight:', [round(w, 3) for w in compute_class_weights(train_df)])

## 3. Full fine-tuning (konfigurasi final Fase 4)
Latih pada **full training set** dengan config final (LR 3e-5, batch 8, **2 epoch**), early stopping (patience=2) + LR scheduler linear+warmup. **2 epoch dipilih** karena learning curve run 3-epoch menunjukkan validation loss minimum di epoch 2 lalu naik di epoch 3 (overfitting) — lihat `outputs/charts/learning_curve_3epoch_justifikasi.png`. Best model (by macro F1) → `models/best_model/`. Loss per epoch diekspor ke `outputs/logs/training_log.csv` untuk learning curve Fase 5.

In [ ]:
from src.modeling.config import TrainingConfig, BEST_MODEL_DIR, ensure_output_dirs
from src.modeling.data import build_hf_dataset, compute_class_weights
from src.modeling.trainer import train_model, export_loss_history_csv
from src.preprocessing.tokenizer_wrapper import IndoBERTTokenizerWrapper

ensure_output_dirs()

# Konfigurasi final: 2 epoch (titik optimal val loss dari learning curve 3-epoch;
# mengurangi overfitting -> gap F1 train vs val). LR 3e-5, batch 8.
final_cfg = TrainingConfig(learning_rate=3e-5, batch_size=8, num_epochs=2, run_name='final')

tokenizer = IndoBERTTokenizerWrapper().tokenizer
class_weights = compute_class_weights(train_df)
train_ds = build_hf_dataset(train_df, tokenizer=tokenizer)
val_ds = build_hf_dataset(val_df, tokenizer=tokenizer)

trainer, val_metrics = train_model(
    final_cfg,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    class_weights=class_weights,
    output_dir=BEST_MODEL_DIR,
    tokenizer=tokenizer,
    save_model=True,
)
print('Validation metrics:', val_metrics)

# Ekspor loss per epoch untuk learning curve Fase 5 (FR-5.6).
log_path = export_loss_history_csv(trainer)
print('Training log ->', log_path)

## 4. Evaluasi test set + cek overfitting (Fase 5, FR-5.1 s/d FR-5.5)
Muat best model dari disk, prediksi test set (metrik final + confusion matrix), serta train & validation untuk cek overfitting (gap F1 train vs val ≤ 5%). Ringkasan 5-fold CV diambil dari report Fase 4.

In [ ]:
from src.evaluation.evaluator import load_best_model, predict_split
from src.evaluation.metrics import build_evaluation_report, compute_classification_metrics, overfitting_gap
from src.evaluation.cross_val_report import summarize_cv

model, tok = load_best_model()

# Prediksi test set -> metrik final + confusion matrix.
y_true, y_pred, y_proba = predict_split(test_df, model, tok)

# Prediksi train & validation -> cek overfitting.
tr_true, tr_pred, _ = predict_split(train_df, model, tok)
va_true, va_pred, _ = predict_split(val_df, model, tok)
train_f1 = compute_classification_metrics(tr_true, tr_pred)['f1_macro']
val_f1 = compute_classification_metrics(va_true, va_pred)['f1_macro']

cv_summary = summarize_cv()
overfit = overfitting_gap(train_f1=train_f1, val_f1=val_f1)

report = build_evaluation_report(y_true, y_pred, cv_summary=cv_summary, overfitting=overfit, write=True)
import json
print(json.dumps(report, indent=2, ensure_ascii=False))

## 5. Confusion matrix (FR-5.5)

In [ ]:
from src.evaluation.visualizer import plot_confusion_matrix

cm_path = plot_confusion_matrix(report['confusion_matrix']['matrix'], normalize=False)
print('Tersimpan:', cm_path)

from IPython.display import Image
Image(str(cm_path))

## 6. Learning curve (FR-5.6)
Training loss vs validation loss per epoch dari `training_log.csv` (Bagian 3).

In [ ]:
from src.evaluation.visualizer import plot_learning_curve

lc_path = plot_learning_curve()
print('Tersimpan:', lc_path)

from IPython.display import Image
Image(str(lc_path))

## 7. Cek gate Fase 5
Validasi: accuracy & macro F1 ≥ 0.85, semua artefak tersedia.

In [ ]:
import subprocess, sys
res = subprocess.run([sys.executable, '-m', 'scripts.validate_phase5_gate'], capture_output=True, text=True)
print(res.stdout)
print(res.stderr)

## 8. Unduh artefak
Arsipkan deliverable Fase 5 (laporan + chart + log) lalu unduh. Taruh isi `outputs/` ke proyek lokal di path yang sama.

In [ ]:
# Artefak Fase 5 (kecil) — laporan + chart + log.
!zip -r artefak_fase5.zip outputs/reports/evaluation_final.json outputs/charts outputs/logs/training_log.csv
from google.colab import files
files.download('artefak_fase5.zip')

# Opsional: arsipkan best model (442 MB) bila ingin menyimpan ulang.
# !zip -r best_model.zip models/best_model
# files.download('best_model.zip')